# 02 — Nettoyage et préparation des données

## Objectif

À partir du dataset brut (`data/raw/movies_raw.csv`), on produit un dataset exploitable pour l'analyse exploratoire et le machine learning.

## Étapes réalisées

1. **Chargement** et parsing des colonnes complexes (listes stockées en string)
2. **Filtrage** des films invalides (budget=0, revenue=0, date manquante, genre absent)
3. **Suppression des doublons** (basée sur l'ID TMDB)
4. **Traitement des valeurs aberrantes** (budget < 1 000$, runtime hors [40-240] min)
5. **Création de features de base** (ROI, is_success, saisonnalité, popularité casting...)
6. **Feature engineering avancé** (historique réalisateur, one-hot genres, inflation)

Le pipeline est encapsulé dans `src/data_cleaning.py` et `src/feature_engineering.py`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from src.data_cleaning import clean_pipeline
from src.feature_engineering import feature_engineering_pipeline

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

## 1. Exploration du dataset brut

In [2]:
df_raw = pd.read_csv(PROJECT_ROOT / 'data' / 'raw' / 'movies_raw.csv')
print(f'Dataset brut : {df_raw.shape}')
df_raw.head(3)

Dataset brut : (4000, 23)


,id,title,original_title,release_date,runtime,budget,revenue,vote_average,vote_count,popularity,original_language,status,genres,production_companies,production_countries,belongs_to_collection,collection_name,director,composer,cast_top5,cast_top5_popularity,overview,tagline
0,10867,Malena,Malèna,2000-10-27,108,0,14493284,7.429,2521,197.6598,it,Released,['Drama'],"['Medusa Film', 'Miramax', 'Pacific Pictures',...","['IT', 'US']",False,NaN,Giuseppe Tornatore,Ennio Morricone,"['Monica Bellucci', 'Giuseppe Sulfaro', 'Lucia...","[13.0921, 0.0, 0.2556, 0.2779, 0.2196]",12-year-old Renato experiences three significa...,"Too young to be a widow, too beautiful to be a..."
1,350,The Devil Wears Prada,The Devil Wears Prada,2006-06-29,109,35000000,326588371,7.397,13279,297.0386,en,Released,"['Drama', 'Comedy']","['Fox 2000 Pictures', 'Wendy Finerman Producti...",['US'],True,The Devil Wears Prada Collection,David Frankel,Theodore Shapiro,"['Meryl Streep', 'Anne Hathaway', 'Emily Blunt...","[8.0172, 27.5047, 12.9007, 7.2937, 5.3784]",A young woman from the Midwest gets more than ...,Meet Andy Sachs. A million girls would kill to...
2,318256,Hot Girls Wanted,Hot Girls Wanted,2015-01-24,84,0,0,6.086,717,202.0650,en,Released,['Documentary'],['Two to Tangle Productions'],['US'],False,NaN,Ronna Gradus,Daniel Ahearn,"['Stella May', 'Brian Omally', 'Ava Taylor', '...","[0.0, 0.0, 0.0, 0.0, 0.0793]",A first-ever look at the realities of the prof...,"Porn, the internet and the girl next door."


In [3]:
# Inventaire des valeurs manquantes ou aberrantes
print('Films avec budget = 0   :', (df_raw['budget'] == 0).sum())
print('Films avec revenue = 0  :', (df_raw['revenue'] == 0).sum())
print('Films sans réalisateur  :', df_raw['director'].isna().sum())
print('Films sans date sortie  :', df_raw['release_date'].isna().sum())
print('Films sans runtime      :', df_raw['runtime'].isna().sum())

Films avec budget = 0   : 876
Films avec revenue = 0  : 628
Films sans réalisateur  : 0
Films sans date sortie  : 0
Films sans runtime      : 0


**Constat** : ~22% des films n'ont pas de budget déclaré, ~16% pas de recettes. Ces films seront filtrés pour permettre les analyses financières et le ML.

## 2. Pipeline de nettoyage

On applique le pipeline complet (voir `src/data_cleaning.py`).

In [4]:
df_clean = clean_pipeline()
print(f'\nDataset nettoyé : {df_clean.shape}')

2026-05-11 11:01:34,568 - INFO - Chargement de C:\Users\lucie\OneDrive - Ynov\Suivi de projet fil rouge\cinevision\data\raw\movies_raw.csv


2026-05-11 11:01:34,597 - INFO - Dataset brut : 4000 films, 23 colonnes


2026-05-11 11:01:34,764 - INFO - Filtrage des films invalides...


2026-05-11 11:01:34,767 - INFO -   Après filtre budget+revenue > 0 : 2979 films (-1021)


2026-05-11 11:01:34,773 - INFO -   Après filtre date valide : 2979 films (-0)


...

2026-05-11 11:01:34,782 - INFO - Total films retenus : 2976 (-3)


2026-05-11 11:01:34,783 - INFO - Création des features de base...


2026-05-11 11:01:34,790 - INFO - Features de base créées : 37 colonnes au total


2026-05-11 11:01:34,860 - INFO - Données nettoyées sauvegardées : C:\Users\lucie\OneDrive - Ynov\Suivi de projet fil rouge\cinevision\data\processed\movies_clean.csv (2976 lignes)



Dataset nettoyé : (2976, 37)


## 3. Feature engineering avancé

Ajout des features dérivées (historique réalisateur, one-hot genres, ajustement inflation, position dans franchise).

In [5]:
df = feature_engineering_pipeline()
print(f'Dataset final : {df.shape}')
print(f'\nColonnes ajoutées : {[c for c in df.columns if c.startswith(("director_", "genre_")) or "2024" in c or c.startswith("franchise_")]}')

2026-05-11 11:01:34,866 - INFO - Chargement de C:\Users\lucie\OneDrive - Ynov\Suivi de projet fil rouge\cinevision\data\processed\movies_clean.csv


2026-05-11 11:01:35,003 - INFO - Calcul des features 'réalisateur'...


2026-05-11 11:01:35,216 - INFO -   Features réalisateur ajoutées


2026-05-11 11:01:35,217 - INFO - Création one-hot encoding pour les 12 genres principaux...


2026-05-11 11:01:35,231 - INFO -   Genres encodés : ['Drama', 'Action', 'Comedy', 'Thriller', 'Adventure', 'Crime', 'Science Fiction', 'Fantasy', 'Horror', 'Family', 'Romance', 'Animation']


2026-05-11 11:01:35,231 - INFO - Ajustement à l'inflation (base 2024)...


2026-05-11 11:01:35,233 - INFO -   Budget et recettes ajustés à l'inflation 2024


2026-05-11 11:01:35,234 - INFO - Features de franchise...


2026-05-11 11:01:35,302 - INFO - Dataset avec features sauvegardé : C:\Users\lucie\OneDrive - Ynov\Suivi de projet fil rouge\cinevision\data\processed\movies_features.csv (2976 lignes)


Dataset final : (2976, 56)

Colonnes ajoutées : ['director_nb_movies_prior', 'director_avg_vote_prior', 'director_avg_roi_prior', 'genre_drama', 'genre_action', 'genre_comedy', 'genre_thriller', 'genre_adventure', 'genre_crime', 'genre_science_fiction', 'genre_fantasy', 'genre_horror', 'genre_family', 'genre_romance', 'genre_animation', 'budget_2024', 'revenue_2024', 'franchise_position']


## 4. Synthèse

In [6]:
print('=== RÉSUMÉ PIPELINE ===')
print(f'  Films bruts     : {len(df_raw)}')
print(f'  Films nettoyés  : {len(df_clean)}')
print(f'  Films finaux    : {len(df)}')
print(f'  Colonnes finales: {df.shape[1]}')
print()
print('=== DISTRIBUTION CIBLE ===')
print(df['is_success'].value_counts())
print(f"  → Taux de succès : {df['is_success'].mean()*100:.1f}%")

=== RÉSUMÉ PIPELINE ===
  Films bruts     : 4000
  Films nettoyés  : 2976
  Films finaux    : 2976
  Colonnes finales: 56

=== DISTRIBUTION CIBLE ===
is_success
1    1961
0    1015
Name: count, dtype: int64
  → Taux de succès : 65.9%
